In [ ]:
!pip install catboost==1.2.3 numpy==1.26.4

# ДЗ
Выполнил Вершинин Данил, Б9122-01.03.02мкт

---

Добавить в Random Forest и Gradient Boosting выбор случайного набора фичей для построения каждого отдельного дерева.

**Необходимо** использовать реализованные деревья из прошлой домашки (реализовать для задачи регрессии)



Основной класс дерева. Чуть переработанный с предыдущего урока

In [ ]:
import numpy as np
import math
import random


class MyRegressionTree:
    def __init__(self, max_depth=5, min_samples_leaf=1, max_features=None):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.root = None

    def _mse(self, y):
        if len(y) == 0:
            return 0
        mean = np.mean(y)
        return np.mean((y - mean) ** 2)

    def _best_split(self, X, y):
        m, n = X.shape
        features = list(range(n))
        if self.max_features is not None:
            features = random.sample(features, self.max_features)

        best_feature, best_threshold, best_mse = None, None, float("inf")
        parent_mse = self._mse(y)

        for feature in features:
            sorted_idx = X[:, feature].argsort()
            X_sorted, y_sorted = X[sorted_idx, feature], y[sorted_idx]

            for i in range(self.min_samples_leaf, m - self.min_samples_leaf):
                if X_sorted[i] == X_sorted[i - 1]:
                    continue
                threshold = (X_sorted[i] + X_sorted[i - 1]) / 2
                y_left, y_right = y_sorted[:i], y_sorted[i:]
                if (
                    len(y_left) < self.min_samples_leaf
                    or len(y_right) < self.min_samples_leaf
                ):
                    continue

                mse_left = self._mse(y_left)
                mse_right = self._mse(y_right)
                mse_total = (len(y_left) * mse_left + len(y_right) * mse_right) / m

                if mse_total < best_mse:
                    best_mse = mse_total
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) <= self.min_samples_leaf:
            return {"value": np.mean(y)}

        feature, threshold = self._best_split(X, y)
        if feature is None:
            return {"value": np.mean(y)}

        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return {
            "feature": feature,
            "threshold": threshold,
            "left": left,
            "right": right,
        }

    def fit(self, X, y):
        self.root = self._build_tree(np.array(X), np.array(y))

    def _predict_one(self, x, node):
        while "value" not in node:
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["value"]

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict_one(x, self.root) for x in X])

добавим лес

In [ ]:
from tqdm.auto import tqdm


class MyRandomForestRegressor:
    def __init__(
        self,
        num_trees=10,
        max_depth=5,
        features_by_tree_strategy="sqrt",
        min_samples_leaf=1,
    ):
        self.num_trees = num_trees
        self.max_depth = max_depth
        self.features_by_tree_strategy = features_by_tree_strategy
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        self.trees = []
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape

        if self.features_by_tree_strategy == "sqrt":
            n_cols_by_tree = int(np.ceil(n_features**0.5))
        elif self.features_by_tree_strategy == "half":
            n_cols_by_tree = int(np.ceil(n_features * 0.5))
        else:
            n_cols_by_tree = n_features

        for i in tqdm(range(self.num_trees)):
            # Bootstrap sample
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = MyRegressionTree(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                max_features=n_cols_by_tree,
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)

# Тоже самое сделать для бустинга

Собственный бустинг

In [ ]:
from tqdm.auto import tqdm


class MyGBDTRegressor:
    def __init__(
        self,
        num_trees=100,
        max_depth=5,
        lr=0.5,
        features_by_tree_strategy="sqrt",
        min_samples_leaf=1,
    ):
        self.num_trees = num_trees
        self.max_depth = max_depth
        self.lr = lr
        self.features_by_tree_strategy = features_by_tree_strategy
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        self.trees = []
        self.init_prediction = np.mean(y)
        self.n_features = X.shape[1]

        if self.features_by_tree_strategy == "sqrt":
            self.max_features = int(np.ceil(self.n_features**0.5))
        elif self.features_by_tree_strategy == "half":
            self.max_features = int(np.ceil(self.n_features * 0.5))
        else:
            self.max_features = self.n_features

        current_prediction = np.full_like(y, self.init_prediction, dtype=np.float64)

        for i in tqdm(range(self.num_trees)):
            residuals = y - current_prediction

            tree = MyRegressionTree(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features,
            )
            tree.fit(X, residuals)
            prediction_update = tree.predict(X)

            current_prediction += self.lr * prediction_update
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X)
        result = np.full(X.shape[0], self.init_prediction, dtype=np.float64)

        for tree in self.trees:
            result += self.lr * tree.predict(X)
        return result

Сравнить результаты от изменения параметра - кол-во фичей для построения дерева. Сравнить с RandomForestRegressor из scikit-learn.

Загрузим датасет

In [ ]:
import pandas as pd

from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

In [ ]:
df = pd.DataFrame(data["data"], columns=data["feature_names"])
df["target"] = data["target"]

In [ ]:
display(df.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


Подготовим выборки

In [ ]:
features = data["feature_names"]
target = "target"

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.25, random_state=41)

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [ ]:
# Вспомогательная функция для фита и рассчёта метрик

global_regression_results = {}


def fit_regression(reg, name):
    global global_regression_results

    reg.fit(X_train.values, y_train.values)
    predict_train = reg.predict(X_train.values)
    predict_test = reg.predict(X_test.values)
    print("Train:")
    train_r2, train_mse, train_spearman = check_metrics(y_train, predict_train)
    print()
    print("Test:")
    test_r2, test_mse, test_spearman = check_metrics(y_test, predict_test)

    global_regression_results[name] = {
        # Train
        "train_r2": train_r2,
        "train_mse": train_mse,
        "train_spearman": train_spearman,
        # Test
        "test_r2": test_r2,
        "test_mse": test_mse,
        "test_spearman": test_spearman,
    }

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

Протестируем модели

In [ ]:
tree = DecisionTreeRegressor(max_depth=8)
fit_regression(tree, "tree")

Train:
Unique Predicts: 232
R2: 0.769
MSE: 0.308
Spearman: 0.867

Test:
Unique Predicts: 194
R2: 0.71
MSE: 0.384
Spearman: 0.844


In [ ]:
my_rf = MyRandomForestRegressor(max_depth=8, num_trees=20)
fit_regression(my_rf, "my_rf")

  0%|          | 0/20 [00:00<?, ?it/s]

Train:
Unique Predicts: 14633
R2: 0.793
MSE: 0.276
Spearman: 0.895

Test:
Unique Predicts: 5039
R2: 0.754
MSE: 0.326
Spearman: 0.876


In [ ]:
my_rf = MyRandomForestRegressor(
    max_depth=8, num_trees=20, features_by_tree_strategy="half"
)
fit_regression(my_rf, "my_rf_half")

  0%|          | 0/20 [00:00<?, ?it/s]

Train:
Unique Predicts: 14156
R2: 0.795
MSE: 0.273
Spearman: 0.89

Test:
Unique Predicts: 4926
R2: 0.756
MSE: 0.323
Spearman: 0.873


In [ ]:
my_rf = MyRandomForestRegressor(
    max_depth=8, num_trees=20, features_by_tree_strategy="all"
)
fit_regression(my_rf, "my_rf_all")

  0%|          | 0/20 [00:00<?, ?it/s]

Train:
Unique Predicts: 11660
R2: 0.803
MSE: 0.262
Spearman: 0.894

Test:
Unique Predicts: 4281
R2: 0.76
MSE: 0.319
Spearman: 0.875


In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf = RandomForestRegressor(max_depth=8, n_estimators=100)
fit_regression(rf, "rf")

Train:
Unique Predicts: 14642
R2: 0.806
MSE: 0.258
Spearman: 0.895

Test:
Unique Predicts: 5011
R2: 0.759
MSE: 0.319
Spearman: 0.875


In [ ]:
my_rf = MyGBDTRegressor(max_depth=8, num_trees=20)
fit_regression(my_rf, "my_gb")

  0%|          | 0/20 [00:00<?, ?it/s]

Train:
Unique Predicts: 15441
R2: 0.935
MSE: 0.087
Spearman: 0.961

Test:
Unique Predicts: 5157
R2: 0.787
MSE: 0.282
Spearman: 0.892


In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
lgbm = LGBMRegressor(n_estimators=50, max_depth=5)
fit_regression(lgbm, "lgbm")

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001592 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 8
[LightGBM] [Info] Start training from score 2.075014
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
pd.DataFrame(global_regression_results).T

,train_r2,train_mse,train_spearman,test_r2,test_mse,test_spearman
tree,0.769,0.308,0.867,0.710,0.384,0.844
my_rf,0.793,0.276,0.895,0.754,0.326,0.876
my_gb,0.935,0.087,0.961,0.787,0.282,0.892
rf,0.806,0.258,0.895,0.759,0.319,0.875
lgbm,0.827,0.231,0.910,0.801,0.264,0.898
my_rf_half,0.795,0.273,0.890,0.756,0.323,0.873
my_rf_all,0.803,0.262,0.894,0.760,0.319,0.875


## Выводы
По итогам тестирования самописный random forest смог обогнать обычное дерево из либы.

С увеличением количества местрик удалось повысить точность на тесте на 5 пунктов.

Использование бустинга дало хороший прирост точности на тесте, но, на мой взгляд, произошло переобучение (0.93 - на трейне).

В сравнении с библиотечными либами результат не сильно отличается.